In [1]:
print("Installing dependencies")
%pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-groq langchain-core faiss-cpu pypdf sentence-transformers python-dotenv langsmith
print("Installation complete. Restart the kernel, then run all cells from the top.")


Installing dependencies
Note: you may need to restart the kernel to use updated packages.
Installation complete. Restart the kernel, then run all cells from the top.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Installing Dependencies 
RAG pipelines:
document loading
embedding model
vector database
llm model

In [2]:
LLM_provider = "groq"
LLM_Model = "openai/gpt-oss-20b"


In [3]:
from pathlib import Path
import os
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f".env not found at {env_path.resolve()}")
load_dotenv(dotenv_path=env_path, override=True)
if not os.getenv("GROQ_API_KEY", "").strip():
    raise RuntimeError(f"GROQ_API_KEY is missing or empty in {env_path.resolve()}")
print("GROQ_API_KEY loaded from .env")


GROQ_API_KEY loaded from .env


In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFDirectoryLoader

CORPUS_PATH = Path("./zyro-dynamics-hr-corpus")
if not CORPUS_PATH.exists():
    raise FileNotFoundError(f"Corpus folder not found: {CORPUS_PATH.resolve()}")

documents = PyPDFDirectoryLoader(str(CORPUS_PATH)).load()
if not documents:
    raise ValueError(f"No readable PDF files found in {CORPUS_PATH.resolve()}")
print(f"Loaded {len(documents)} document pages")


C:\Users\hp\AppData\Local\Temp\ipykernel_13396\1009109108.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 39 document pages


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")


Created 107 chunks


##Embedding
HuggingfacesEmbedding

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings initialized")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1519.36it/s]


Embeddings initialized


VectorDB + Reterival
 
 FAISS Vector DB
 

In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("FAISS vector store and retriever initialized")


FAISS vector store and retriever initialized


## LLM Initilaziation
LLM model -groq

In [8]:
from langchain_groq import ChatGroq

llm_model = ChatGroq(model=LLM_Model, temperature=0.7, max_tokens=500)
print(f"Groq model {LLM_Model} initialized")


Groq model openai/gpt-oss-20b initialized


# RAG CHAIN 
 initilization


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable 
RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are an HR assistant. Answer using only the context below.
      If the answer is not in the context,
        say you do not have that information.

Context:
{context}

Question: {question}
Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = RAG_PROMPT | llm_model | StrOutputParser()

def ask_hr(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    answer = rag_chain.invoke({"context": context, "question": question})
    return {"answer": answer, "sources": docs}


Guardials

to ensure that no prompts used beyond corups


In [10]:
GUARDIAL_PROMPT = ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant. Decide whether the question below is something 
an HR assistant should answer (company leave policy, reimbursement, code of conduct, or other 
internal HR topics) or something out of scope (general knowledge, coding help,
 unrelated small talk, etc.).

Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.

Question: {question}
""")

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)

def ask_bot(question: str):
    guardrail_chain = GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict = guardrail_chain.invoke({"question": question}).strip().upper()

    if "OUT_OF_SCOPE" in verdict:
        return {"answer": REFUSAL_MESSAGE, "sources": []}

    return ask_hr(question)

print("Guardial Initialized!")

Guardial Initialized!


In [11]:
test_question = [
    "How many casual leaves do I get per year?",  # related to the corpus
    "Q01,How does my Earned Leave accrue every month?",
    "Q04,Do I need a medical certificate for sick leave?",
     "Q05,What date does my salary get credited every month?"
    "Q06,What's the salary range for an L4 Senior employee?",      # related to the corpus
    "What's the capital of France?",              # not related to the corpus
]

for i, q in enumerate(test_question, 1):
    print(f"\nQ{i}: {q}")

    try:
        result = ask_bot(q)
        print(f"A{i}: {result['answer']}")

    except Exception as e:
        print(f"Error in Q{i}: {type(e).__name__}: {e}")


Q1: How many casual leaves do I get per year?
A1: I do not have that information.

Q2: Q01,How does my Earned Leave accrue every month?
A2: Earned Leave (EL) is added to your balance each month based on how long you’ve been with the company:

* **After you’ve completed one year of continuous service** –

Q3: Q04,Do I need a medical certificate for sick leave?
A3: Yes.  
If your sick leave is **more than 2 consecutive days**, you must obtain a medical certificate from a registered medical practitioner and submit it within 3 working days of your return to work. (Sick leave of 2 days or fewer does not require a certificate.)

Q4: Q05,What date does my salary get credited every month?Q06,What's the salary range for an L4 Senior employee?
A4: **Q05** – Your salary is credited to your registered bank account by the **7th of the following month**.  

**Q06** – The salary range for an L4 Senior employee is **Rs. 16.0 L to Rs. 26.0 L per annum**.

Q5: What's the capital of France?
A5: I'm an H

In [12]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Read your CSV file

In [13]:
import pandas as pd

test_df = pd.read_csv("test.csv")

print(test_df.head())
print(test_df.columns)

  question_id                                           question
0         Q01       How does my Earned Leave accrue every month?
1         Q02  How much Earned Leave can I carry forward to n...
2         Q03  How many weeks of Maternity Leave am I entitle...
3         Q04    Do I need a medical certificate for sick leave?
4         Q05  What date does my salary get credited every mo...
Index(['question_id', 'question'], dtype='str')


In [15]:
import pandas as pd

df = pd.read_csv("test.csv")

print(df.head())

  question_id                                           question
0         Q01       How does my Earned Leave accrue every month?
1         Q02  How much Earned Leave can I carry forward to n...
2         Q03  How many weeks of Maternity Leave am I entitle...
3         Q04    Do I need a medical certificate for sick leave?
4         Q05  What date does my salary get credited every mo...


Generate answers for all questions

In [16]:
results = []

for i, row in df.iterrows():
    question = row["question"]

    try:
        response = ask_bot(question)
        answer = response["answer"]

    except Exception as e:
        answer = f"Error: {e}"

    results.append({
        "question_id": row["question_id"],
        "question": question,
        "answer": answer
    })

    print(f"Completed: {row['question_id']}")

Completed: Q01
Completed: Q02
Completed: Q03
Completed: Q04
Completed: Q05
Completed: Q06
Completed: Q07
Completed: Q08
Completed: Q09
Completed: Q10
Completed: Q11
Completed: Q12
Completed: Q13
Completed: Q14
Completed: Q15
Completed: Q16
Completed: Q17
Completed: Q18
Completed: Q19
Completed: Q20


In [ ]:
answers_df = pd.DataFrame(results)

answers_df.to_csv("rag_answers.csv", index=False)

print("All answers saved successfully!")

All answers saved successfully!


In [ ]:
submission = answers_df[["question_id", "answer"]]

submission.to_csv("submission.csv", index=False)

submission.head()

,question_id,answer
0,Q01,Earned Leave (EL) accrues at a fixed rate of *...
1,Q02,I do not have that information.
2,Q03,You are entitled to **26 weeks of paid Materni...
3,Q04,"Yes, but only if your sick leave exceeds 2 con..."
4,Q05,Your salary is credited by the **7th of the fo...
